In [1]:
!pip install -q pennylane pennylane-lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 42.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 60.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 96.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.9 MB/s eta 0:00:00


In [2]:
#!/usr/bin/env python3
"""
================================================================================
RFQCL — Quantum Continual Modulation Recognition (Idea 1)
RadioML 2016.10a · continual learning across degrading SNRs · Mamba + VQC head
================================================================================

ONE-CELL experiment suite for Kaggle (paste the whole file into one cell).

What it runs (top-tier-paper grade, per the Research Paper Operating Law):
  - 2 continual protocols: SNR-incremental (5 tasks, high->low SNR: "the
    channel degrades") and class-incremental (4 tasks adding modulation
    classes).
  - 8 methods x 3 seeds:
      mamba_mlp        Mamba encoder + MLP head (naive sequential)
      cnn_mlp          CNN encoder  + MLP head (naive sequential)
      mamba_vqc        Mamba encoder + variational-quantum head
      mamba_vqc_ewc    mamba_vqc + Elastic Weight Consolidation
      cnn_vqc          CNN encoder  + VQC head  (isolates the quantum head)
      mamba_mlp_er     mamba_mlp + experience replay (reservoir buffer)
      mamba_mlp_lwf    mamba_mlp + Learning-without-Forgetting distillation
      joint            all tasks at once (upper-bound reference only)
  - Metrics: per-task accuracy matrix, avg_acc, forgetting, backward transfer,
    accuracy-vs-SNR, trainable params, wall time; summary reports mean +/- std
    over seeds and paired significance tests.
  - Both GPUs are used simultaneously (one worker thread per GPU). Quantum
    heads run on the CPU (lightning.qubit) with 2 OpenMP threads each, so two
    quantum units run concurrently without oversubscribing the CPU.

ACCUMULATIVE CACHE (survives Kaggle session wipes)
  Every finished unit is written to its own results_<hash>.json AND merged into
  a single cumulative results.json. On startup the code merges everything it
  finds — the working cache and any cache attached from a previous session's
  saved output under /kaggle/input — so results accumulate across sessions.
  After each run, "Save Version" to persist the cache; next session attach that
  version as an Input dataset and re-run.

DATA (real benchmark; no fabricated data)
  RadioML 2016.10a (O'Shea & West, GNU Radio channel simulations) is the
  field-standard modulation-classification benchmark. Downloaded once from its
  Zenodo mirror (zenodo.org/records/18397070) and cached. The signals are
  simulated-at-origin by the benchmark authors (standard AMC practice). 11
  modulations x 20 SNRs (-20..+18 dB) x 1000 IQ samples (2 x 128).

FAIRNESS (same conditions for every method, per the Fair-Baseline Law)
  Same dataset subset, same class/SNR task splits, same 3 seeds, same AdamW +
  lr, same epochs, same batch size, same metric code, same eval protocol,
  same grad clip. Encoders are size-matched (Mamba ~45k, CNN ~46k params).
  ER and LwF share the exact mamba_mlp architecture; cnn_vqc shares the
  cnn_mlp encoder and the mamba_vqc head. Trainable parameter counts are
  reported per method. Replay/LwF/joint hyper-parameters are printed in CONFIG.

1-HOUR SCALE (reduced-sample regime)
  subset_train_per_cell=100 with epochs=2 and single-pass evaluation keeps the
  full 48-unit suite around one hour on 2xT4 while staying a recognized
  limited-sample AMC setting. Use --full (500/class, 8 epochs) to scale up.

Flags: --full (500/500, 8 epochs) | --quick (smoke) | --tiny (QA) | --force
       (ignore cache) | --selfbreak (negative control, QA) | --gpus N
================================================================================
"""

import argparse
import copy
import hashlib
import json
import math
import os
import pickle
import random
import shutil
import sys
import tarfile
import threading
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import pennylane as qml

# =============================================================================
# CONFIG (edit here; no CLI required for the research knobs)
# =============================================================================
CONFIG = dict(
    # --- data (reduced-sample regime: a recognized limited-sample AMC setting) ---
    subset_train_per_cell=100,   # train samples per (mod,snr) cell (500 = classic split)
    subset_test_per_cell=100,    # test  samples per (mod,snr) cell
    # --- training (equal for every method => fair) ---
    epochs=2,                    # epochs for classical-head methods
    batch=256,                   # batch for classical-head methods
    q_epochs=2,                  # epochs for quantum-head methods (equal => fair)
    q_batch=256,                 # batch  for quantum-head methods (equal => fair)
    lr=1e-3,
    grad_clip=1.0,               # global grad-norm clip (stability; all methods)
    seeds=[42, 7, 123],          # exactly 3 seeds
    # --- quantum head ---
    n_qubits=4,
    q_layers=2,
    # --- encoders (matched in size => fair) ---
    mamba=dict(d_model=48, d_state=8, n_layers=2),   # ~45k params
    cnn_width=64,                                   # CNN ~46k params (matched)
    # --- continual learning ---
    ewc_lambda=50.0,             # EWC regularization strength (mamba_vqc_ewc)
    er_buffer_per_class=20,      # replay exemplars per class (mamba_mlp_er)
    lwf_lambda=1.0,              # distillation weight (mamba_mlp_lwf)
    lwf_T=2.0,                   # distillation temperature (mamba_mlp_lwf)
    # --- joint upper bound (offline reference; documented budget) ---
    joint_epochs=5,              # a few more epochs: it is the offline upper bound
    joint_lr_scale=0.5,          # half LR for stability on the full mixed-SNR set
    # --- protocols ---
    protocols=["SNR_inc", "class_inc"],
    methods=["mamba_mlp", "cnn_mlp", "mamba_vqc", "mamba_vqc_ewc",
             "cnn_vqc", "mamba_mlp_er", "mamba_mlp_lwf", "joint"],
    # --- cache ---
    cache_dir=os.environ.get("RFQCL_CACHE", "/kaggle/working/rfqcl_cache"),
)

# Zenodo mirror of RadioML 2016.10a (verified 2026-08-13: HTTP 200, 212 MB)
DATASET_URL = "https://zenodo.org/records/18397070/files/RML2016.10a.tar.bz2?download=1"
PICKLE_NAME = "RML2016.10a_dict_optimized.pkl"

# SNR-incremental tasks: 5 tasks x 4 SNRs, ordered high SNR -> low SNR
# (the channel degrades as the stream advances).
SNR_TASKS = [[18, 16, 14, 12], [10, 8, 6, 4], [2, 0, -2, -4],
             [-6, -8, -10, -12], [-14, -16, -18, -20]]
# Class-incremental tasks: 4 tasks over the 11 modulation classes.
CLASS_TASKS = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10]]

MOD_NAMES = ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK',
             'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']

# Thread-safety for the multi-GPU (thread) scheduler.
SEED_LOCK = threading.Lock()     # serializes model construction + global seeding
PRINT_LOCK = threading.Lock()    # keeps each unit's printed block contiguous
RESULTS_LOCK = threading.Lock()  # serializes the cumulative results.json update


def is_quantum(method):
    """True for any method with a variational-quantum head."""
    return "vqc" in method


def banner(text, ch="="):
    print(f"\n{ch * 80}\n{text}\n{ch * 80}", flush=True)


# =============================================================================
# Data loading (real benchmark, cached)
# =============================================================================
def download_radioml(cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    tar_path = os.path.join(cache_dir, "RML2016.10a.tar.bz2")
    pkl_path = os.path.join(cache_dir, PICKLE_NAME)
    if not os.path.exists(pkl_path):
        if not os.path.exists(tar_path):
            print(f"[data] downloading RadioML 2016.10a (212 MB) from Zenodo ...", flush=True)
            t0 = time.time()
            urllib.request.urlretrieve(DATASET_URL, tar_path)
            print(f"[data] downloaded in {time.time() - t0:.1f}s", flush=True)
        print("[data] extracting ...", flush=True)
        with tarfile.open(tar_path, "r:bz2") as tf:
            try:
                tf.extractall(cache_dir, filter="data")   # silences the 3.14 warning
            except TypeError:                              # older Pythons
                tf.extractall(cache_dir)
    return pickle.load(open(pkl_path, "rb"), encoding="latin1")


def build_subset(Xd, n_train, n_test, cache_dir):
    """Random balanced subset per (mod,snr) cell; cached to disk."""
    cache_path = os.path.join(cache_dir, f"dataset_t{n_train}_v{n_test}.pt")
    if os.path.exists(cache_path):
        return torch.load(cache_path, weights_only=False)

    snrs = sorted(set(snr for _, snr in Xd.keys()))
    mods = sorted(set(mod for mod, _ in Xd.keys()))
    mod2id = {m: i for i, m in enumerate(mods)}
    rng = np.random.RandomState(20260813)
    Xtr, Xte, ytr, yte, strr, ste = [], [], [], [], [], []
    for mod in mods:
        for snr in snrs:
            cell = Xd[(mod, snr)]                    # (1000, 2, 128)
            idx = rng.permutation(cell.shape[0])
            tr = cell[idx[:n_train]]
            te = cell[idx[n_train:n_train + n_test]]
            # per-sample unit-power normalization (standard AMC preprocessing)
            tr = tr / (np.sqrt(np.mean(tr ** 2, axis=(1, 2), keepdims=True)) + 1e-9)
            te = te / (np.sqrt(np.mean(te ** 2, axis=(1, 2), keepdims=True)) + 1e-9)
            Xtr.append(tr); Xte.append(te)
            ytr += [mod2id[mod]] * n_train
            yte += [mod2id[mod]] * n_test
            strr += [snr] * n_train
            ste += [snr] * n_test
    Xtr = np.concatenate(Xtr); Xte = np.concatenate(Xte)
    ytr = np.array(ytr); yte = np.array(yte)
    strr = np.array(strr); ste = np.array(ste)
    print(f"[data] subset built: train {Xtr.shape[0]} / test {Xte.shape[0]} "
          f"({n_train}+{n_test} per (mod,snr) cell)", flush=True)
    torch.save((Xtr, ytr, strr, Xte, yte, ste, mods, snrs), cache_path)
    return Xtr, ytr, strr, Xte, yte, ste, mods, snrs


def make_loader(X, y, idxs, batch, shuffle, device, rng=None):
    """Dataset whose index order can be re-shuffled per epoch (thread-safe).

    The Dataset holds a shared view of the full X/y tensors plus a mutable
    order list. Training reshuffles the order each epoch via the per-unit
    RandomState (never the global RNG), so no two GPU threads race.
    """
    idxs = list(idxs)
    x_t = torch.from_numpy(np.asarray(X)).float()
    y_t = torch.from_numpy(np.asarray(y)).long()

    class DS(torch.utils.data.Dataset):
        def __init__(self):
            self.order = idxs[:]

        def reshuffle(self, rng):
            rng.shuffle(self.order)

        def __len__(self):
            return len(self.order)

        def __getitem__(self, i):
            j = self.order[i]
            return x_t[j], y_t[j]

    ds = DS()
    if shuffle:
        (np.random.shuffle(ds.order) if rng is None else rng.shuffle(ds.order))
        shuffle = False
    return torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=shuffle,
                                       num_workers=0, pin_memory=(device == "cuda"))


# =============================================================================
# Selective scan + Mamba block (pure PyTorch, bidirectional)
# =============================================================================
def selective_scan(x, dt, A, B, C, Dv, chunk=8):
    """Chunked selective scan (exact, vectorized within chunks, numerically
    stable). Within a chunk the recurrence is solved with the associative
    (cumsum) closed form, exact for time-invariant A; the running state is
    carried across chunks. Chunking keeps the exponential numerically stable.
    """
    B_, L, Dd = x.shape
    N = B.shape[-1]
    h = torch.zeros(B_, Dd, N, device=x.device, dtype=x.dtype)
    A01 = A[None, None, :, :]                                 # (1,1,D,N)
    D01 = Dv[None, None, :]
    ys = []
    for s in range(0, L, chunk):
        xc = x[:, s:s + chunk]
        dtc = dt[:, s:s + chunk]
        Sloc = torch.cumsum(dtc, dim=1)                       # (B,c,D)
        dBc = dtc[:, :, :, None] * B[:, s:s + chunk, None, :] * xc[:, :, :, None]  # (B,c,D,N)
        ASloc = A01 * Sloc[:, :, :, None]                     # (B,c,D,N)
        cum = torch.cumsum(torch.exp(-ASloc) * dBc, dim=1)    # (B,c,D,N)
        h = torch.exp(ASloc) * (h[:, None, :, :] + cum)       # (B,c,D,N)
        ys.append(torch.einsum("bln,bldn->bld", C[:, s:s + chunk], h) + D01 * xc)
        h = h[:, -1]                                          # carry state
    return torch.cat(ys, dim=1)


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state, dt_rank, expand=2, d_conv=3, selfbreak=False):
        super().__init__()
        self.selfbreak = selfbreak
        self.dt_rank = dt_rank
        self.d_state = d_state
        d_inner = int(expand * d_model)
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, 2 * d_inner, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, d_conv, groups=d_inner, padding=d_conv - 1)
        self.x_proj = nn.Linear(d_inner, dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        dt = torch.exp(torch.rand(d_inner) * (math.log(0.1) - math.log(1.0)) + math.log(1.0))
        self.dt_bias = nn.Parameter(dt - F.softplus(dt))
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def _mix(self, x):
        x = x.transpose(1, 2)
        x = self.conv1d(x)[:, :, : x.shape[2]]
        x = F.silu(x).transpose(1, 2)
        if self.selfbreak:
            x = x[torch.arange(x.shape[0]) - 1]
        xdb = self.x_proj(x)
        dt, B, C = torch.split(xdb, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt) + self.dt_bias)
        dt = dt.clamp(max=1.0)   # bound Delta for the chunked scan's stability
        A = -torch.exp(self.A_log)
        return selective_scan(x, dt, A, B, C, self.D)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x, z = xz.chunk(2, dim=-1)
        y = self._mix(x) + torch.flip(self._mix(torch.flip(x, [1])), [1])
        y = self.out_proj(y * F.silu(z))
        return residual + y


class MambaEncoder(nn.Module):
    """(B, 2, T) -> mean-pooled features (B, d_model)."""
    def __init__(self, d_model=48, d_state=8, n_layers=2, T=128, selfbreak=False):
        super().__init__()
        self.T = T
        self.embed = nn.Linear(2, d_model, bias=False)
        self.pos = nn.Parameter(torch.zeros(1, T, d_model))
        self.blocks = nn.ModuleList([
            MambaBlock(d_model, d_state, dt_rank=max(4, d_state // 2), selfbreak=selfbreak)
            for _ in range(n_layers)
        ])
        self.norm_f = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x.transpose(1, 2)                       # (B, T, 2)
        x = self.embed(x) + self.pos
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_f(x)
        return x.mean(dim=1)                        # (B, d_model)


class CNNEncoder(nn.Module):
    """(B, 2, T) -> features (B, 2*width). VTCNN-style 1-D CNN, widened so its
    parameter count matches the Mamba encoder (equal-size comparison)."""
    def __init__(self, width=64):
        super().__init__()
        self.width = width
        self.net = nn.Sequential(
            nn.Conv1d(2, width, 3, padding=1), nn.ReLU(),
            nn.Conv1d(width, width, 3, padding=1), nn.ReLU(),
            nn.Conv1d(width, 2 * width, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x):
        return self.net(x).flatten(1)


class MLPHead(nn.Module):
    def __init__(self, d_in, n_classes, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_classes))

    def forward(self, f):
        return self.net(f)


class VQCHead(nn.Module):
    """Variational-quantum head: features -> angle-encoded VQC -> Z expvals -> logits.

    Uses lightning.qubit (CPU) with adjoint differentiation + native batch
    broadcasting: the whole mini-batch goes through the circuit in one call.
    """
    def __init__(self, d_in, n_classes, n_qubits=4, q_layers=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.q_layers = q_layers
        self.proj = nn.Linear(d_in, n_qubits)
        self.q_weights = nn.Parameter(torch.randn(q_layers, n_qubits) * 0.1)
        self.out = nn.Linear(n_qubits, n_classes)
        dev = qml.device("lightning.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="adjoint")
        def qnode(features, weights):
            for L in range(q_layers):
                for w in range(n_qubits):
                    qml.RY(features[:, w], wires=w)
                for w in range(n_qubits - 1):
                    qml.CNOT(wires=[w, w + 1])
                for w in range(n_qubits):
                    qml.RY(weights[L, w], wires=w)
            return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]

        self.qnode = qnode

    def forward(self, f):
        dev = f.device
        if f.is_cuda:
            f = f.cpu()                                   # lightning.qubit runs on CPU
        feats = torch.tanh(self.proj(f)) * math.pi        # angles in [-pi, pi]
        exp = torch.stack(self.qnode(feats, self.q_weights), dim=1)  # (B, n_qubits)
        return self.out(exp).to(dev)

    def _apply(self, fn, recurse=True):
        # Keep this head pinned to CPU no matter what device/dtype the parent
        # module is moved to (Module.to()/cuda()/float() recurse via _apply,
        # so returning self without applying fn keeps the quantum parameters on
        # CPU, which lightning.qubit requires; forward handles the transfer).
        return self


def build_model(method, n_classes, cfg, selfbreak=False):
    if method.startswith("cnn"):
        w = cfg.get("cnn_width", 64)
        enc = CNNEncoder(width=w); d_feat = 2 * w
    else:
        enc = MambaEncoder(d_model=cfg["mamba"]["d_model"],
                           d_state=cfg["mamba"]["d_state"],
                           n_layers=cfg["mamba"]["n_layers"], selfbreak=selfbreak)
        d_feat = cfg["mamba"]["d_model"]
    if is_quantum(method):
        head = VQCHead(d_feat, n_classes, n_qubits=cfg["n_qubits"],
                       q_layers=cfg["q_layers"])
    else:
        head = MLPHead(d_feat, n_classes)
    return nn.Sequential(enc, head)


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# =============================================================================
# Metrics (standard continual-learning definitions)
# =============================================================================
def cl_metrics(acc_matrix):
    """acc_matrix: (n_tasks, n_tasks) where [t, j] = accuracy on task j after
    learning task t (NaN if not yet evaluated)."""
    T = acc_matrix.shape[0]
    final = acc_matrix[T - 1]
    avg_acc = float(np.nanmean(final))
    forgets = []
    for j in range(T - 1):
        hist = acc_matrix[:, j]
        peak = float(np.nanmax(hist))
        forgets.append(peak - float(final[j]))
    forgetting = float(np.mean(forgets))
    bwt = float(np.mean([float(final[j]) - float(acc_matrix[j, j]) for j in range(T - 1)]))
    return dict(avg_acc=avg_acc, forgetting=forgetting, bwt=bwt)


def label_remap(seen, n_classes, device):
    remap = torch.full((n_classes,), -1, dtype=torch.long, device=device)
    remap[torch.tensor(seen, device=device)] = torch.arange(len(seen), device=device)
    return remap


# =============================================================================
# EWC + replay buffer + distillation
# =============================================================================
class EWC:
    def __init__(self, model, lam):
        self.model = model
        self.lam = lam
        self.fisher = {}
        self.star = {}

    def update(self, loader, seen, n_classes, device):
        model = self.model
        model.train()
        grads = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
        remap = label_remap(seen, n_classes, device)
        n_batches = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            model.zero_grad()
            logits = model(xb)[:, seen]
            loss = F.cross_entropy(logits, remap[yb])
            loss.backward()
            n_batches += 1
            for n, p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    grads[n] += p.grad.detach() ** 2
        for n in grads:
            grads[n] /= max(n_batches, 1)
            if n in self.fisher:
                self.fisher[n] += grads[n]
            else:
                self.fisher[n] = grads[n]
        self.star = {n: p.detach().clone() for n, p in model.named_parameters()
                     if p.requires_grad}

    def penalty(self):
        # Accumulate on ONE device (hybrid models have CPU head + GPU encoder)
        # while keeping each term differentiable for the EWC gradient.
        pen = None
        for n, p in self.model.named_parameters():
            if n in self.fisher:
                term = (self.fisher[n] * (p - self.star[n]) ** 2).sum()
                pen = term if pen is None else pen + term.to(pen.device)
        return 0.5 * self.lam * (pen if pen is not None else 0.0)


class ReplayBuffer:
    """Fixed reservoir of exemplars, up to per_class per class."""
    def __init__(self, per_class):
        self.per_class = per_class
        self.x = None   # (M, 2, 128) float32
        self.y = None   # (M,) int64

    def add(self, X, y, idxs, classes, rng):
        new_x, new_y = [], []
        for c in classes:
            ci = [i for i in idxs if y[i] == c]
            if ci:
                keep = ci[:self.per_class]
                new_x.append(X[keep])
                new_y.extend([y[i] for i in keep])
        if not new_x:
            return
        xb = torch.from_numpy(np.concatenate(new_x)).float()
        yb = torch.tensor(new_y, dtype=torch.long)
        self.x = xb if self.x is None else torch.cat([self.x, xb])
        self.y = yb if self.y is None else torch.cat([self.y, yb])

    def sample(self, n, rng):
        if self.x is None or self.x.shape[0] == 0:
            return None
        n = min(n, self.x.shape[0])
        idx = torch.from_numpy(rng.choice(self.x.shape[0], size=n, replace=False))
        return self.x[idx], self.y[idx]


def distill_loss(new_logits, old_logits, T):
    p = F.log_softmax(new_logits / T, dim=1)
    q = F.softmax(old_logits / T, dim=1)
    return F.kl_div(p, q, reduction="batchmean") * (T * T)


# =============================================================================
# Training + evaluation
# =============================================================================
def train_epoch(model, opt, loader, seen, n_classes, device, ewc=None,
                reshuffle_rng=None, replay=None, lwf=None, grad_clip=0.0):
    if reshuffle_rng is not None:
        loader.dataset.reshuffle(reshuffle_rng)
    model.train()
    remap = label_remap(seen, n_classes, device)
    tot = corr = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits_full = model(xb)
        logits = logits_full[:, seen]
        loss = F.cross_entropy(logits, remap[yb])
        if ewc is not None:
            loss = loss + ewc.penalty()
        if replay is not None and reshuffle_rng is not None:
            rp = replay.sample(xb.shape[0], reshuffle_rng)
            if rp is not None:
                bx, by = rp
                bx, by = bx.to(device), by.to(device)
                rlog = model(bx)[:, seen]
                loss = loss + F.cross_entropy(rlog, remap[by])
        if lwf is not None and lwf["snapshot"] is not None:
            with torch.no_grad():
                old_logits = lwf["snapshot"](xb)[:, lwf["old_classes"]]
            new_logits = logits_full[:, lwf["old_classes"]]
            loss = loss + lwf["lam"] * distill_loss(new_logits, old_logits, lwf["T"])
        loss.backward()
        if grad_clip > 0:
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],
                                     grad_clip)
        opt.step()
        corr += (logits.argmax(1) == remap[yb]).sum().item()
        tot += yb.shape[0]
    return corr / max(tot, 1)


@torch.no_grad()
def evaluate(model, loader, seen, n_classes, device):
    model.eval()
    n = len(seen)
    corr = torch.zeros(n)
    tot = torch.zeros(n)
    remap = label_remap(seen, n_classes, device)
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)[:, seen]
        pred = logits.argmax(1)
        ybr = remap[yb]
        for li in range(n):
            m = ybr == li
            corr[li] += (pred[m] == li).sum().item()
            tot[li] += m.sum().item()
    return {seen[i]: (corr[i] / max(tot[i], 1)).item() for i in range(n)}


@torch.no_grad()
def evaluate_counts(model, Xte, yte, ste, idxs, batch, seen, n_classes, device, snr_list):
    """ONE pass over the test set: returns per-(snr, class) correct/total counts.

    From these counts the caller derives, with the exact same macro metrics as
    before: per-task accuracy (mean over classes of class accuracy on a group
    of SNRs) and per-SNR accuracy. Replaces the ~35 separate passes the old
    code needed, which is what made quantum units slow at evaluation time.
    """
    model.eval()
    remap = label_remap(seen, n_classes, device)
    n = len(seen)
    corr = {int(s): torch.zeros(n, device=device) for s in snr_list}
    tot = {int(s): torch.zeros(n, device=device) for s in snr_list}
    for b0 in range(0, len(idxs), batch):
        bi = idxs[b0:b0 + batch]
        xb = torch.from_numpy(np.asarray(Xte)[bi]).float().to(device)
        yb = torch.from_numpy(np.asarray(yte)[bi]).long().to(device)
        sb = torch.from_numpy(np.asarray(ste)[bi]).long().to(device)
        logits = model(xb)[:, seen]
        pred = logits.argmax(1)
        ybr = remap[yb]
        for li in range(n):
            m = (ybr == li)
            for s in snr_list:
                ms = m & (sb == s)
                corr[int(s)][li] += (pred[ms] == li).sum()
                tot[int(s)][li] += ms.sum()
    C = {s: corr[s].cpu().numpy().astype(float) for s in corr}
    T = {s: tot[s].cpu().numpy().astype(float) for s in tot}
    return C, T


def run_snr_inc(model, data, cfg, method, device, seed):
    Xtr, ytr, strr, Xte, yte, ste, mods, snrs = data
    n_classes = len(mods)
    tasks = SNR_TASKS
    rng = np.random.RandomState(seed)   # per-unit, thread-safe shuffle source
    n_ep = cfg["q_epochs"] if is_quantum(method) else cfg["epochs"]
    batch = cfg["q_batch"] if is_quantum(method) else cfg["batch"]
    clip = cfg.get("grad_clip", 0.0)
    train_idxs = lambda t: [i for i, s in enumerate(strr) if s in tasks[t]]
    all_test = list(range(len(yte)))
    all_seen = list(range(n_classes))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg["lr"])
    ewc = EWC(model, cfg["ewc_lambda"]) if method == "mamba_vqc_ewc" else None
    replay = ReplayBuffer(cfg["er_buffer_per_class"]) if method == "mamba_mlp_er" else None
    lwf = (dict(lam=cfg["lwf_lambda"], T=cfg["lwf_T"], snapshot=None,
                old_classes=list(range(n_classes)))
           if method == "mamba_mlp_lwf" else None)
    acc_mat = np.full((len(tasks), len(tasks)), np.nan)
    snr_acc = {}
    for t in range(len(tasks)):
        with PRINT_LOCK:
            print(f"    [{method}] task {t+1}/{len(tasks)} training", flush=True)
        tr = make_loader(Xtr, ytr, train_idxs(t), batch, True, device, rng=rng)
        for ep in range(n_ep):
            acc = train_epoch(model, opt, tr, all_seen, n_classes, device, ewc=ewc,
                              reshuffle_rng=rng, replay=replay, lwf=lwf, grad_clip=clip)
            with PRINT_LOCK:
                print(f"    [{method}] task {t+1}/{len(tasks)} epoch {ep+1}/{n_ep} "
                      f"train_acc={acc:.3f}", flush=True)
        if ewc is not None:
            ewc.update(make_loader(Xtr, ytr, train_idxs(t), batch, False, device),
                       all_seen, n_classes, device)
        if replay is not None:
            replay.add(Xtr, ytr, train_idxs(t), all_seen, rng)
        if lwf is not None:
            lwf["snapshot"] = copy.deepcopy(model)
            lwf["snapshot"].eval()
        # One test pass -> per-task and per-SNR accuracy (macro over classes).
        C, Tcnt = evaluate_counts(model, Xte, yte, ste, all_test, batch,
                                  all_seen, n_classes, device, snrs)
        for j in range(t + 1):
            g = [int(s) for s in tasks[j]]
            num = sum(C[s] for s in g)
            den = sum(Tcnt[s] for s in g)
            acc_mat[t, j] = float(np.mean(num / np.maximum(den, 1)))
        if t == len(tasks) - 1:
            snr_acc = {int(s): float(np.mean(C[s] / np.maximum(Tcnt[s], 1))) for s in snrs}
    m = cl_metrics(acc_mat)
    return dict(acc_matrix=acc_mat.round(4).tolist(), **{k: round(v, 4) for k, v in m.items()},
                snr_acc=snr_acc)


def run_class_inc(model, data, cfg, method, device, seed):
    Xtr, ytr, strr, Xte, yte, ste, mods, snrs = data
    n_classes = len(mods)
    tasks = CLASS_TASKS
    rng = np.random.RandomState(seed)   # per-unit, thread-safe shuffle source
    n_ep = cfg["q_epochs"] if is_quantum(method) else cfg["epochs"]
    batch = cfg["q_batch"] if is_quantum(method) else cfg["batch"]
    train_idxs = lambda t: [i for i, c in enumerate(ytr) if c in tasks[t]]
    test_idxs = lambda t: [i for i, c in enumerate(yte) if c in tasks[t]]
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg["lr"])
    ewc = EWC(model, cfg["ewc_lambda"]) if method == "mamba_vqc_ewc" else None
    replay = ReplayBuffer(cfg["er_buffer_per_class"]) if method == "mamba_mlp_er" else None
    lwf = (dict(lam=cfg["lwf_lambda"], T=cfg["lwf_T"], snapshot=None, old_classes=[])
           if method == "mamba_mlp_lwf" else None)
    acc_mat = np.full((len(tasks), len(tasks)), np.nan)
    for t in range(len(tasks)):
        seen = [c for ts in tasks[:t + 1] for c in ts]
        with PRINT_LOCK:
            print(f"    [{method}] task {t+1}/{len(tasks)} training", flush=True)
        tr = make_loader(Xtr, ytr, train_idxs(t), batch, True, device, rng=rng)
        for ep in range(n_ep):
            acc = train_epoch(model, opt, tr, seen, n_classes, device, ewc=ewc,
                              reshuffle_rng=rng, replay=replay, lwf=lwf,
                              grad_clip=cfg.get("grad_clip", 0.0))
            with PRINT_LOCK:
                print(f"    [{method}] task {t+1}/{len(tasks)} epoch {ep+1}/{n_ep} "
                      f"train_acc={acc:.3f}", flush=True)
        if ewc is not None:
            ewc.update(make_loader(Xtr, ytr, train_idxs(t), batch, False, device),
                       seen, n_classes, device)
        if replay is not None:
            replay.add(Xtr, ytr, train_idxs(t), tasks[t], rng)
        if lwf is not None:
            lwf["snapshot"] = copy.deepcopy(model)
            lwf["snapshot"].eval()
            lwf["old_classes"] = seen
        # Single evaluation over ALL classes seen so far, then per-task mean
        # (evaluating only task-j data with all seen classes would count absent
        # classes as 0.0 and dilute the per-task numbers).
        all_test = []
        for j in range(t + 1):
            all_test += test_idxs(j)
        accs = evaluate(model, make_loader(Xte, yte, all_test, batch, False, device),
                        seen, n_classes, device)
        for j in range(t + 1):
            acc_mat[t, j] = float(np.mean([accs[c] for c in tasks[j]]))
    m = cl_metrics(acc_mat)
    return dict(acc_matrix=acc_mat.round(4).tolist(), **{k: round(v, 4) for k, v in m.items()})


def run_joint(model, data, cfg, device, proto, seed):
    Xtr, ytr, strr, Xte, yte, ste, mods, snrs = data
    n_classes = len(mods)
    rng = np.random.RandomState(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"] * cfg.get("joint_lr_scale", 0.5))
    if proto == "SNR_inc":
        tasks, train_idx, test_idx = SNR_TASKS, \
            lambda t: [i for i, s in enumerate(strr) if s in SNR_TASKS[t]], \
            lambda t: [i for i, s in enumerate(ste) if s in SNR_TASKS[t]]
        seen = list(range(n_classes))
    else:
        tasks, train_idx, test_idx = CLASS_TASKS, \
            lambda t: [i for i, c in enumerate(ytr) if c in CLASS_TASKS[t]], \
            lambda t: [i for i, c in enumerate(yte) if c in CLASS_TASKS[t]]
        seen = [c for ts in CLASS_TASKS for c in ts]
    all_tr = []
    for t in range(len(tasks)):
        all_tr += train_idx(t)
    loader = make_loader(Xtr, ytr, all_tr, cfg["batch"], True, device, rng=rng)
    # Joint upper bound (offline reference): trained a few more epochs at half
    # LR on the full mixed-SNR set, with gradient clipping for stability. It is
    # NOT a fair sequential comparison target - it is the achievable ceiling.
    for ep in range(cfg.get("joint_epochs", 5)):
        acc = train_epoch(model, opt, loader, seen, n_classes, device, reshuffle_rng=rng,
                          grad_clip=cfg.get("grad_clip", 0.0))
        with PRINT_LOCK:
            print(f"    [joint] epoch {ep+1}/{cfg['epochs']} train_acc={acc:.3f}", flush=True)
    acc_mat = np.full((len(tasks), len(tasks)), np.nan)
    for t in range(len(tasks)):
        seen_t = list(range(n_classes)) if proto == "SNR_inc" else tasks[t]
        accs = evaluate(model, make_loader(Xte, yte, test_idx(t), cfg["batch"], False, device),
                        seen_t, n_classes, device)
        acc_mat[len(tasks) - 1, t] = float(np.mean(list(accs.values())))
    m = cl_metrics(acc_mat)
    m["bwt"] = 0.0   # joint has no sequential-transfer story
    return dict(acc_matrix=acc_mat.round(4).tolist(),
                **{k: round(v, 4) for k, v in m.items()})


# =============================================================================
# Self-test (runs before the main suite; must be fast; QA only)
# =============================================================================
def selftest(cfg, selfbreak=False):
    torch.manual_seed(0); np.random.seed(0)
    # 1. Mamba encoder: shapes, gradient flow, per-sample isolation
    enc = MambaEncoder(d_model=32, d_state=8, n_layers=2, T=64, selfbreak=selfbreak)
    x = torch.randn(4, 2, 64)
    y = enc(x)
    assert y.shape == (4, 32), y.shape
    y.sum().backward()
    assert enc.embed.weight.grad is not None
    with torch.no_grad():
        xp = x.clone(); xp[0] += torch.randn(2, 64) * 0.5
        d = (enc(xp) - enc(x)).abs()
        assert d[1:].max().item() < 1e-5, "samples coupled (scan isolation broken)"
        assert d[0].max().item() > 1e-4, "sample 0 not affected"
    # 2. scan linearity (fixed parameters)
    with torch.no_grad():
        B_, L, Dd, N = 3, 12, 32, 8
        dt = torch.rand(B_, L, Dd); A = -torch.exp(torch.rand(Dd, N))
        Bm = torch.rand(B_, L, N); C = torch.rand(B_, L, N); Dv = torch.rand(Dd)
        a, b = torch.randn(B_, L, Dd), torch.randn(B_, L, Dd)
        fab = selective_scan(a + b, dt, A, Bm, C, Dv) \
              - (selective_scan(a, dt, A, Bm, C, Dv) + selective_scan(b, dt, A, Bm, C, Dv))
        assert fab.abs().max().item() < 1e-3, "scan not linear"
    # 3. VQC head: forward shape + gradient through the quantum layer
    vh = VQCHead(32, 11, n_qubits=cfg["n_qubits"], q_layers=cfg["q_layers"])
    f = torch.randn(8, 32)
    out = vh(f)
    assert out.shape == (8, 11), out.shape
    out.sum().backward()
    assert vh.q_weights.grad is not None and vh.proj.weight.grad is not None
    # 4. quantum readout depends on features (negative-control target)
    with torch.no_grad():
        o1 = vh(torch.zeros(8, 32)); o2 = vh(torch.randn(8, 32))
        assert (o1 - o2).abs().max().item() > 1e-3, "VQC blind to features"
    # 5. metrics hand-check
    A = np.array([[0.9, np.nan], [0.5, 0.7]])
    m = cl_metrics(A)
    assert abs(m["avg_acc"] - 0.6) < 1e-6 and abs(m["forgetting"] - 0.4) < 1e-6, m
    # 6. new method construction paths
    for meth in ["cnn_vqc", "mamba_mlp_er", "mamba_mlp_lwf", "cnn_mlp", "mamba_vqc"]:
        mdl = build_model(meth, 11, cfg)
        with torch.no_grad():
            o = mdl(torch.randn(2, 2, 128))
        assert o.shape == (2, 11), (meth, o.shape)
    # 7. replay buffer + distillation sanity
    rb = ReplayBuffer(2)
    X = np.random.rand(6, 2, 128).astype(np.float32)
    yv = np.array([0, 0, 1, 1, 2, 2])
    rb.add(X, yv, list(range(6)), [0, 1, 2], np.random.RandomState(0))
    bx, by = rb.sample(4, np.random.RandomState(1))
    assert bx.shape == (4, 2, 128) and by.shape == (4,)
    dl = distill_loss(torch.randn(4, 5), torch.randn(4, 5), 2.0)
    assert float(dl) >= 0.0
    # 8. cache round-trip
    import tempfile
    d = tempfile.mkdtemp()
    p = os.path.join(d, "c.json")
    json.dump({"a": [1.0, 2.0]}, open(p, "w"))
    assert json.load(open(p))["a"][1] == 2.0
    print("[selftest] PASS: encoder shapes/grad/isolation, scan linearity, "
          "VQC forward/grad/feature-sensitivity, metrics, all method builders, "
          "replay/distill, cache IO")


# =============================================================================
# Accumulative cache (per-unit files + one cumulative master; merge across
# sessions from /kaggle/input)
# =============================================================================
MASTER_NAME = "results.json"


def config_signature(cfg):
    return hashlib.md5(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:8]


def unit_file(cache_dir, uid):
    h = hashlib.md5(uid.encode()).hexdigest()[:12]
    return os.path.join(cache_dir, f"results_{h}.json")


def master_path(cache_dir):
    return os.path.join(cache_dir, MASTER_NAME)


def write_unit(cache_dir, uid, sig, r):
    """Write a finished unit to its own file AND merge into the cumulative
    master results.json. This makes the cache accumulative: the master always
    contains every unit ever finished, so attaching one file is enough to
    resume across Kaggle session wipes."""
    uf = unit_file(cache_dir, uid)
    json.dump({"uid": uid, "cfg_sig": sig, "result": r}, open(uf, "w"))
    with RESULTS_LOCK:
        master = {}
        mp = master_path(cache_dir)
        if os.path.exists(mp):
            try:
                master = json.load(open(mp))
            except Exception:
                master = {}
        master[uid] = {"cfg_sig": sig, "result": r}
        tmp = mp + ".tmp"
        json.dump(master, open(tmp, "w"))
        os.replace(tmp, mp)


def _read_master_into(results, path, sig):
    try:
        for k, v in json.load(open(path)).items():
            if isinstance(v, dict) and "result" in v:
                if sig is not None and v.get("cfg_sig") != sig:
                    continue
                results[k] = v["result"]
            elif isinstance(v, dict) and "|" in k:      # legacy value-only format
                if sig is not None and v.get("cfg_sig", "") != sig:
                    continue
                results[k] = v
    except Exception:
        pass


def load_all_results(cache_dir, sig=None):
    """Union of everything available: the cumulative master and all per-unit
    files in the cache dir. `sig` filters to one config (stale configs are
    never reused as if they were the current one)."""
    results = {}
    _read_master_into(results, master_path(cache_dir), sig)
    for fn in sorted(os.listdir(cache_dir)):
        if fn.startswith("results_") and fn.endswith(".json"):
            try:
                blob = json.load(open(os.path.join(cache_dir, fn)))
                if sig is not None and blob.get("cfg_sig") != sig:
                    continue
                results[blob["uid"]] = blob["result"]
            except Exception:
                pass
    return results


def find_attached_caches():
    """Locate caches from a previous Kaggle session under /kaggle/input."""
    found = []
    base = "/kaggle/input"
    if not os.path.isdir(base):
        return found
    try:
        for root, dirs, files in os.walk(base):
            depth = root[len(base):].count(os.sep)
            if depth > 3:
                dirs[:] = []
                continue
            for d in list(dirs):
                if d == "rfqcl_cache":
                    found.append(os.path.join(root, d))
            for fn in files:
                if (fn.startswith("results_") and fn.endswith(".json")) \
                        or fn == MASTER_NAME:
                    found.append(root)
    except Exception:
        pass
    seen, out = set(), []
    for p in found:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def merge_attached_caches(cache_dir):
    """Copy result files AND expand attached masters into per-unit files, so
    everything ever produced accumulates into the working cache."""
    for src in find_attached_caches():
        try:
            merged = 0
            for fn in sorted(os.listdir(src)):
                if fn.startswith("results_") and fn.endswith(".json"):
                    dst = os.path.join(cache_dir, fn)
                    if not os.path.exists(dst):
                        shutil.copy2(os.path.join(src, fn), dst)
                        merged += 1
                elif fn == MASTER_NAME:
                    dst = os.path.join(cache_dir, MASTER_NAME)
                    if not os.path.exists(dst):
                        shutil.copy2(os.path.join(src, fn), dst)
                        merged += 1
                    # also expand any units that only exist inside the master
                    try:
                        for k, v in json.load(open(os.path.join(src, fn))).items():
                            if isinstance(v, dict) and "result" in v:
                                uf = unit_file(cache_dir, k)
                                if not os.path.exists(uf):
                                    json.dump({"uid": k, "cfg_sig": v.get("cfg_sig", ""),
                                               "result": v["result"]}, open(uf, "w"))
                    except Exception:
                        pass
            if merged:
                print(f"[cache] merged {merged} file(s) from attached cache: {src}", flush=True)
        except Exception as e:
            print(f"[cache] could not merge attached cache {src}: {e}", flush=True)


# =============================================================================
# Suite driver (accumulative cache + parallel GPUs + full printing)
# =============================================================================
def _worker(gpu_id, my_units, cfg, cache_dir, data, force, sig):
    """Run one worker's slice of the queue on a single GPU (or the CPU).

    Runs in a THREAD (never a forked process): a notebook kernel already has
    CUDA initialized, and forking after CUDA init fails. Threads share the
    kernel's CUDA context; each pins its own GPU. Quantum heads run on the CPU
    (lightning.qubit) with OMP_NUM_THREADS=2 set in main(), so two concurrent
    quantum units use 4 CPU threads total without oversubscription - both GPUs
    stay busy at the same time.
    """
    if gpu_id is not None:
        torch.cuda.set_device(gpu_id)
        torch.backends.cudnn.benchmark = False
        try:
            torch.backends.cudnn.deterministic = True
        except Exception:
            pass
    device = f"cuda:{gpu_id}" if gpu_id is not None else "cpu"
    for (proto, method, seed) in my_units:
        uid = f"{proto}|{method}|{seed}"
        tag = (f"[gpu:{gpu_id}]" if gpu_id is not None else "[cpu]") + f" {uid}"
        uf = unit_file(cache_dir, uid)
        if os.path.exists(uf) and not force:
            try:
                blob = json.load(open(uf))
                if blob.get("cfg_sig") == sig:
                    with PRINT_LOCK:
                        _print_unit(tag + "  [CACHED]", proto, method, blob["result"])
                    continue
            except Exception:
                pass
        with PRINT_LOCK:
            print(f"\n>>> STARTING {tag}", flush=True)
        t0 = time.time()
        try:
            with SEED_LOCK:
                torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
                model = build_model("mamba_mlp" if method == "joint" else method,
                                    len(MOD_NAMES), cfg).to(device)
            if method == "joint":
                r = run_joint(model, data, cfg, device, proto, seed)
            elif proto == "SNR_inc":
                r = run_snr_inc(model, data, cfg, method, device, seed)
            else:
                r = run_class_inc(model, data, cfg, method, device, seed)
            r["n_params"] = n_params(model)
            r["wall_s"] = round(time.time() - t0, 1)
            r["gpu"] = -1 if gpu_id is None else gpu_id
            write_unit(cache_dir, uid, sig, r)
            with PRINT_LOCK:
                _print_unit(tag, proto, method, r)
                print(f"[saved] {uid} -> {uf}")
        except Exception as e:
            with PRINT_LOCK:
                print(f"[ERROR] {uid} failed: {type(e).__name__}: {e}", flush=True)


def run_suite(cfg, force=False, gpus=None):
    banner("RFQCL — Quantum Continual Modulation Recognition (RadioML 2016.10a)")
    print("CONFIG:")
    for k, v in cfg.items():
        print(f"  {k} = {v}")
    cache_dir = cfg["cache_dir"]
    try:
        os.makedirs(cache_dir, exist_ok=True)
    except OSError:
        cache_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                 os.path.basename(cache_dir.rstrip("/")) or "rfqcl_cache")
        os.makedirs(cache_dir, exist_ok=True)
        print(f"[cache] fallback dir: {cache_dir}")
    # Accumulate: pull in caches attached from previous sessions, then union
    # everything (per-unit files + cumulative master).
    merge_attached_caches(cache_dir)
    sig = config_signature(cfg)
    results = load_all_results(cache_dir, sig)
    all_results = load_all_results(cache_dir, None)

    # ---- data (loaded once; worker THREADS share it in-process) ----
    Xd = download_radioml(cache_dir)
    data = build_subset(Xd, cfg["subset_train_per_cell"], cfg["subset_test_per_cell"], cache_dir)

    # ---- queue ----
    units = []
    for proto in cfg["protocols"]:
        for method in cfg["methods"]:
            for seed in cfg["seeds"]:
                units.append((proto, method, seed))
    n_cached = sum(1 for p, m, s in units if f"{p}|{m}|{s}" in results)
    print(f"\nexperiment queue: {len(units)} units | cached (this config): {n_cached} "
          f"| to run: {len(units) - n_cached}")
    print(f"[cache] accumulative store: {len(all_results)} result(s) on disk across all configs")
    if n_cached == 0:
        print("  note: no matching cache. Normal on a NEW Kaggle session (Kaggle wipes")
        print("        /kaggle/working). To resume: Save Version at the end of a run,")
        print("        then next session Add Input -> Datasets -> Your Work -> select it.")
    elif n_cached == len(units):
        print("  note: every unit is cached; nothing to run. --force recomputes.")

    # ---- GPUs ----
    if gpus is None:
        try:
            gpus = torch.cuda.device_count()
        except Exception:
            gpus = 0
    gpus = max(0, int(gpus))
    print(f"\ndevice: {gpus} GPU(s) | torch {torch.__version__} | pennylane {qml.__version__}")
    print(f"config signature: {sig} (cache is invalidated automatically if config changes)")

    # ---- pre-registered expectations ----
    print("\npre-registered expectations (CHECK after run):")
    print("  - joint (upper bound) avg_acc should exceed every sequential method.")
    print("  - mamba_vqc_ewc forgetting should be <= mamba_vqc forgetting.")
    print("  - quantum heads (mamba_vqc) should forget less than MLP heads (mamba_mlp).")
    print("  - mamba_mlp_er / mamba_mlp_lwf should forget less than mamba_mlp.")
    print("  - high-SNR tasks should show higher accuracy than low-SNR tasks.")

    t0_all = time.time()
    if gpus <= 1:
        _worker(0 if gpus == 1 else None, units, cfg, cache_dir, data, force, sig)
    else:
        n_threads = min(gpus, len(units))
        with ThreadPoolExecutor(max_workers=n_threads) as ex:
            futs = [ex.submit(_worker, g, units[g::n_threads], cfg, cache_dir, data, force, sig)
                    for g in range(n_threads)]
            for f in futs:
                f.result()    # propagate any unexpected (non-unit) error
    print(f"\ntotal wall time: {(time.time() - t0_all) / 60:.1f} min")
    _print_summary(load_all_results(cache_dir, sig), cfg)
    print("\n" + "=" * 70)
    print("TO KEEP THIS CACHE FOR YOUR NEXT KAGGLE SESSION:")
    print("  1. Click 'Save Version' (top-right). The accumulative cache lives in")
    print(f"     {cache_dir} and is captured by the saved version.")
    print("  2. Next session: Add Input -> Datasets -> Your Work -> select that saved")
    print("     version.")
    print("  3. Re-run this cell. The code merges the attached cache and resumes from")
    print("     the first unfinished unit (no recompute).")
    print("=" * 70)


def _print_unit(tag, proto, method, r):
    print("\n" + "-" * 70)
    print(f"UNIT {tag}")
    if "acc_matrix" in r:
        print("  acc_matrix (rows=tasks learned so far, cols=task test):")
        for row in r["acc_matrix"]:
            print("    " + " ".join(f"{v if not (isinstance(v, float) and v != v) else '  .':>7}"
                                    for v in row))
    print(f"  avg_acc={r.get('avg_acc')}  forgetting={r.get('forgetting')}  "
          f"bwt={r.get('bwt', '-')}  n_params={r.get('n_params')}  wall_s={r.get('wall_s')}")
    if "snr_acc" in r:
        print("  accuracy vs SNR (final model, all classes):")
        items = sorted(r["snr_acc"].items(), key=lambda kv: -int(kv[0]))
        print("    " + " ".join(f"{s}={a:.3f}" for s, a in items))


def _paired_t(a, b):
    """Paired t-test on two lists of per-seed values (scipy if available)."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    d = a - b
    n = len(d)
    if n < 2:
        return None
    mean = float(d.mean()); sd = float(d.std(ddof=1))
    t = mean / (sd / math.sqrt(n) + 1e-12)
    p = None
    try:
        from scipy import stats as _st
        p = float(2 * _st.t.sf(abs(t), n - 1))
    except Exception:
        pass
    return dict(t=t, p=p, mean_diff=mean)


def _print_summary(results, cfg):
    banner("SUMMARY (all methods x protocols, mean +/- std over seeds)")
    for proto in cfg["protocols"]:
        print(f"\n  protocol = {proto}")
        print(f"  {'method':<16}{'avg_acc':>16}{'forgetting':>16}{'bwt':>10}{'params':>9}")
        for method in cfg["methods"]:
            rows = [results[f"{proto}|{method}|{s}"] for s in cfg["seeds"]
                    if f"{proto}|{method}|{s}" in results]
            if not rows:
                print(f"  {method:<16}{'missing':>16}")
                continue
            acc = np.array([r["avg_acc"] for r in rows])
            fgt = np.array([r["forgetting"] for r in rows])
            bwt = np.array([r.get("bwt", 0) for r in rows])
            prm = rows[0].get("n_params", 0)
            print(f"  {method:<16}{acc.mean():>8.4f}±{acc.std(ddof=1):<6.4f}"
                  f"{fgt.mean():>8.4f}±{fgt.std(ddof=1):<6.4f}"
                  f"{bwt.mean():>10.4f}{prm:>9d}")
    print("\n  (joint is the upper-bound reference; it violates the CL protocol.)")

    # paired significance over the 3 seeds, per protocol
    print("\n" + "-" * 70)
    print("PAIRED SIGNIFICANCE (over seeds, two-sided t-test)")
    for proto in cfg["protocols"]:
        def vec(method, key):
            return [results[f"{proto}|{method}|{s}"][key] for s in cfg["seeds"]
                    if f"{proto}|{method}|{s}" in results]
        pairs = [
            ("mamba_vqc", "mamba_mlp", "forgetting", "quantum head forgets less than MLP head"),
            ("mamba_vqc_ewc", "mamba_vqc", "forgetting", "EWC reduces forgetting"),
            ("mamba_mlp_er", "mamba_mlp", "forgetting", "replay reduces forgetting"),
            ("mamba_mlp_lwf", "mamba_mlp", "forgetting", "LwF reduces forgetting"),
            ("cnn_vqc", "cnn_mlp", "forgetting", "quantum head forgets less (CNN encoder)"),
        ]
        for m1, m2, key, desc in pairs:
            v1, v2 = vec(m1, key), vec(m2, key)
            if len(v1) >= 2 and len(v2) >= 2:
                t = _paired_t(v1, v2)
                ps = "n/a" if t["p"] is None else f"{t['p']:.4f}"
                print(f"  [{proto}] {desc}: {m1} {np.mean(v1):.4f} vs {m2} "
                      f"{np.mean(v2):.4f} | t={t['t']:+.2f}, p={ps}")
            else:
                print(f"  [{proto}] {desc}: SKIP (missing seeds)")

    # pre-registered expectation checks (reported, not enforced)
    print("\n" + "-" * 70)
    print("PRE-REGISTERED EXPECTATION CHECKS")
    for proto in cfg["protocols"]:
        def agg(method, key):
            rows = [results[f"{proto}|{method}|{s}"][key] for s in cfg["seeds"]
                    if f"{proto}|{method}|{s}" in results]
            return float(np.mean(rows)) if rows else float("nan")
        joint = agg("joint", "avg_acc")
        best_seq = max(agg(m, "avg_acc") for m in cfg["methods"] if m != "joint")
        if np.isnan(joint) or np.isnan(best_seq):
            print(f"  [{proto}] joint-vs-sequential: SKIP (missing results)")
        else:
            print(f"  [{proto}] joint={joint:.4f} vs best sequential={best_seq:.4f} -> "
                  f"{'OK' if joint >= best_seq - 1e-6 else 'CHECK (unexpected)'}")
        f_ewc, f_vqc = agg("mamba_vqc_ewc", "forgetting"), agg("mamba_vqc", "forgetting")
        if np.isnan(f_ewc) or np.isnan(f_vqc):
            print(f"  [{proto}] EWC-vs-noEWC: SKIP (missing results)")
        else:
            print(f"  [{proto}] EWC forgetting={f_ewc:.4f} vs no-EWC={f_vqc:.4f} -> "
                  f"{'OK (EWC <= no-EWC)' if f_ewc <= f_vqc + 1e-6 else 'CHECK (EWC did not help)'}")

    print("\n" + "-" * 70)
    print("FAIRNESS / PROTOCOL NOTES (for the paper's setup section)")
    print("  - All methods: same subset, same class/SNR splits, same 3 seeds,")
    print("    same AdamW(lr=%s), same epochs=%s, same batch=%s, same metric code," % (
        cfg["lr"], cfg["epochs"], cfg["batch"]))
    print("    same eval protocol, same grad clip=%.2f. Quantum heads use" % cfg["grad_clip"])
    print("    q_epochs=%s / q_batch=%s (kept equal for fairness)." % (
        cfg["q_epochs"], cfg["q_batch"]))
    print("  - Encoders are size-matched: Mamba ~45k params, CNN (width=%d) ~46k." %
          cfg["cnn_width"])
    print("    The VQC head has far fewer trainable params than the MLP head; this is")
    print("    reported per method and is a conservative position for the quantum method.")
    print("  - joint: offline upper bound, %d epochs at %.1fx LR (documented; not a" % (
        cfg["joint_epochs"], cfg["joint_lr_scale"]))
    print("    fair sequential target).")
    print("  - mamba_mlp_er: reservoir replay, %d exemplars per class (disclosed)." %
          cfg["er_buffer_per_class"])
    print("  - mamba_mlp_lwf: distillation, lambda=%.2f, T=%.1f (disclosed)." %
          (cfg["lwf_lambda"], cfg["lwf_T"]))
    print("  - cnn_vqc shares the cnn_mlp encoder and the mamba_vqc head; ER/LwF")
    print("    share the exact mamba_mlp architecture. Trainable params are reported")
    print("    per method in the table above.")


def _filter_argv(argv):
    """Strip '-f <connection-file>' that IPython/Colab %run injects."""
    out, skip_next = [], False
    for a in argv:
        if skip_next:
            skip_next = False
            continue
        if a == "-f":
            skip_next = True
            continue
        out.append(a)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--full", action="store_true")
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--tiny", action="store_true")
    ap.add_argument("--selfbreak", action="store_true")
    ap.add_argument("--force", action="store_true")
    ap.add_argument("--gpus", type=int, default=None,
                    help="number of GPUs to use (default: auto-detect all)")
    args = ap.parse_args(_filter_argv(sys.argv[1:]))
    # Two OpenMP + two torch CPU threads: quantum heads run on the CPU, and
    # this lets two quantum units run concurrently (one per GPU thread) using
    # ~4 CPU threads total - no oversubscription, both GPUs busy.
    os.environ["OMP_NUM_THREADS"] = "2"
    try:
        torch.set_num_threads(2)
    except Exception:
        pass
    cfg = dict(CONFIG)
    if args.tiny:
        cfg.update(subset_train_per_cell=40, subset_test_per_cell=20, epochs=1,
                   q_epochs=1, seeds=[42], batch=64, q_batch=64,
                   cache_dir=cfg["cache_dir"] + "_tiny",
                   mamba=dict(d_model=32, d_state=4, n_layers=2))
    if args.quick:
        cfg.update(subset_train_per_cell=60, subset_test_per_cell=30, epochs=2,
                   q_epochs=2, seeds=[42], cache_dir=cfg["cache_dir"] + "_quick")
    if args.full:
        cfg.update(subset_train_per_cell=500, subset_test_per_cell=500, epochs=8,
                   q_epochs=8)
    if args.selfbreak:
        try:
            selftest(cfg, selfbreak=True)
        except AssertionError as e:
            print(f"[selfbreak] NEGATIVE CONTROL PASSED: selftest failed as expected -> {e}")
            return 0
        print("[selfbreak] FAILED: selftest did not catch the defect (check inadequate)")
        return 1
    print("running self-test ...")
    selftest(cfg, selfbreak=False)
    run_suite(cfg, force=args.force, gpus=args.gpus)
    return 0


if __name__ == "__main__":
    # Detect the notebook OUTSIDE main(), so a NameError raised inside main()
    # can never be swallowed and cause a second run.
    try:
        get_ipython()
        in_notebook = True
    except NameError:
        in_notebook = False
    if in_notebook:
        main()          # no sys.exit -> no fake "SystemExit" in the notebook
    else:
        sys.exit(main())


running self-test ...
[selftest] PASS: encoder shapes/grad/isolation, scan linearity, VQC forward/grad/feature-sensitivity, metrics, all method builders, replay/distill, cache IO

RFQCL — Quantum Continual Modulation Recognition (RadioML 2016.10a)
CONFIG:
  subset_train_per_cell = 100
  subset_test_per_cell = 100
  epochs = 2
  batch = 256
  q_epochs = 2
  q_batch = 256
  lr = 0.001
  grad_clip = 1.0
  seeds = [42, 7, 123]
  n_qubits = 4
  q_layers = 2
  mamba = {'d_model': 48, 'd_state': 8, 'n_layers': 2}
  cnn_width = 64
  ewc_lambda = 50.0
  er_buffer_per_class = 20
  lwf_lambda = 1.0
  lwf_T = 2.0
  joint_epochs = 5
  joint_lr_scale = 0.5
  protocols = ['SNR_inc', 'class_inc']
  methods = ['mamba_mlp', 'cnn_mlp', 'mamba_vqc', 'mamba_vqc_ewc', 'cnn_vqc', 'mamba_mlp_er', 'mamba_mlp_lwf', 'joint']
  cache_dir = /kaggle/working/rfqcl_cache

experiment queue: 48 units | cached (this config): 0 | to run: 48
[cache] accumulative store: 0 result(s) on disk across all configs
  note: no mat